<a href="https://colab.research.google.com/github/devharsh/cyberquest-camp/blob/main/Day1/Day1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Day 1 Notebook: Python First Steps

Run a cell with **Shift + Enter**. Work through the Modules in order.

**Modules in this notebook:**
1. Module 1: Variables and data types
2. Module 2: Loops
3. Module 3: Functions
4. Module 4: Password strength checker (challenge)

The slides tell you when to run each Module. After each Module, go back to the slides.


## Module 1: Variables and data types
A variable stores a value. Python has strings (text), integers (whole numbers), floats (decimals), and booleans (True/False).


In [ ]:
agent = "Ada"
missions = 3
ready = True
print(agent, "has", missions, "missions. Ready?", ready)
print(type(agent), type(missions), type(ready))


## Module 2: Loops
A `for` loop repeats work. This is the same idea as moving a hero through a maze in CodeCombat.


In [ ]:
for n in range(1, 6):
    print("Mission", n, "ready")


## Module 3: Functions
A function is a reusable block of code. Define it once with `def`, then call it many times.


In [ ]:
def greet(name):
    return "Welcome, " + name + "!"

print(greet("Ada"))
print(greet("Sam"))


## Module 4: Password strength checker (your challenge)
Score a password: +1 for length 12 or more, +1 for a digit, +1 for an uppercase letter, +1 for a symbol.


In [ ]:
def strength(pw):
    score = 0
    if len(pw) >= 12: score += 1
    if any(c.isdigit() for c in pw): score += 1
    if any(c.isupper() for c in pw): score += 1
    if any(not c.isalnum() for c in pw): score += 1
    return ["Weak","Weak","Okay","Strong","Strong"][score]

for pw in ["sunshine", "Sunshine1", "purple-kite-river-stone-7"]:
    print(pw, "->", strength(pw))


## Linux terminal reference (for OverTheWire Bandit)
You will use these in the OverTheWire Bandit game:
- `ls` list files, `ls -la` show all with detail
- `cd folder` move in, `cd ..` move up, `pwd` where am I
- `cat file` print a file
- `ssh user@host -p 2220` log in to another computer (SSH = Secure Shell)


## Going further: encoding and the network

Building on Modules 1 to 4, these add encoding skills and basic network tools. Standard library only; network or root-only cells are clearly marked as concepts.

### String Manipulation: encode/decode, hex, slicing

Strings and bytes are different in Python 3 -- this trips up beginners.
Encoding converts str -> bytes; decoding goes bytes -> str.


In [ ]:
msg = 'Hello, CyberQuest!'

# Encode to bytes
msg_bytes = msg.encode('utf-8')
print('Encoded bytes:', msg_bytes)

# Hex representation -- very common in security
msg_hex = msg_bytes.hex()
print('Hex string:  ', msg_hex)

# Decode hex back
recovered = bytes.fromhex(msg_hex).decode('utf-8')
print('Recovered:   ', recovered)

# String slicing -- useful for parsing log lines
log_line = '2024-06-01 14:23:11 FAILED login from 10.0.0.5'
date     = log_line[:10]          # first 10 chars
ip_part  = log_line.split()[-1]   # last token
print(f'Date: {date}  |  Source IP: {ip_part}')
# Expected: Date: 2024-06-01  |  Source IP: 10.0.0.5


### TCP Port Scanner with Threading


In [ ]:
import socket, threading
from queue import Queue

def check_port(host: str, port: int, open_ports: list, timeout: float = 0.5):
    """Check if a single TCP port is open."""
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(timeout)
            result = s.connect_ex((host, port))
            if result == 0:
                open_ports.append(port)
    except (socket.timeout, ConnectionRefusedError, OSError):
        pass

def threaded_port_scan(host: str, ports: list, max_threads: int = 50) -> list:
    """Scan multiple ports concurrently using threads."""
    open_ports = []
    threads    = []
    for port in ports:
        t = threading.Thread(target=check_port, args=(host, port, open_ports))
        threads.append(t)
        t.start()
        # Limit concurrent threads
        if len([th for th in threads if th.is_alive()]) >= max_threads:
            for th in threads:
                th.join(timeout=0.1)
    for t in threads:
        t.join()
    return sorted(open_ports)

# Demo: scan localhost for common ports
# (likely closed in this environment, which is expected)
common_ports = [22, 80, 443, 3306, 5432, 6379, 8080, 8443]
print('Scanning localhost for common ports...')
open_p = threaded_port_scan('127.0.0.1', common_ports)
print(f'Open ports: {open_p if open_p else "(none found -- expected in sandbox)"}')


### DNS Lookup


In [ ]:
import socket

def dns_lookup(hostname: str) -> str:
    """Resolve a hostname to an IP address."""
    try:
        return socket.gethostbyname(hostname)
    except socket.gaierror as e:
        return f'Error: {e}'

def reverse_dns(ip: str) -> str:
    """Reverse DNS -- IP to hostname."""
    try:
        return socket.gethostbyaddr(ip)[0]
    except socket.herror as e:
        return f'Error: {e}'

domains = ['google.com', 'github.com', 'stevens.edu']
for d in domains:
    ip = dns_lookup(d)
    print(f'{d:20s} -> {ip}')


### Traceroute Concept

Traceroute maps the path packets take to a destination by sending packets
with increasing TTL (Time To Live) values. Each hop decrements TTL by 1;
when TTL hits 0, the router sends back an ICMP 'time exceeded' message.


In [ ]:
import subprocess, platform

def ping_host(host: str) -> bool:
    """Ping a host once. Returns True if reachable."""
    flag = '-n' if platform.system() == 'Windows' else '-c'
    result = subprocess.run(
        ['ping', flag, '1', '-W', '1', host],
        capture_output=True, text=True
    )
    return result.returncode == 0

# Concept demo -- real traceroute requires root/admin on most systems
print('Traceroute concept:')
print('  TTL=1: packet expires at first router (your gateway)')
print('  TTL=2: packet expires at second hop')
print('  ...and so on until destination is reached')
print()
print('Pinging 8.8.8.8 (Google DNS)...')
alive = ping_host('8.8.8.8')
print(f'8.8.8.8 reachable: {alive}')


### HTTP vs HTTPS Request Comparison


In [ ]:
import urllib.request, ssl

def fetch_headers(url: str) -> dict:
    """Fetch HTTP headers from a URL."""
    try:
        ctx = ssl.create_default_context()
        with urllib.request.urlopen(url, timeout=5, context=ctx) as r:
            return dict(r.headers)
    except Exception as e:
        return {'error': str(e)}

print('Security headers to look for in HTTPS responses:')
security_headers = [
    ('Strict-Transport-Security', 'Enforces HTTPS only'),
    ('Content-Security-Policy',   'Prevents XSS attacks'),
    ('X-Frame-Options',           'Prevents clickjacking'),
    ('X-Content-Type-Options',    'Prevents MIME sniffing'),
    ('Referrer-Policy',           'Controls referrer info'),
]
for header, purpose in security_headers:
    print(f'  {header:<35} -- {purpose}')

print('\nNote: To test, call fetch_headers("https://example.com")')
print('and check which security headers are present or missing.')


### Simple Packet Sniffer Concept

**Note:** A real packet sniffer requires root/admin privileges and a library
like `scapy`. We describe the concept here for educational purposes.


In [ ]:
# Packet sniffer concept (requires root -- do NOT run in production)
SNIFFER_CONCEPT = '''
from scapy.all import sniff, IP, TCP

def packet_callback(packet):
    if IP in packet and TCP in packet:
        src = packet[IP].src
        dst = packet[IP].dst
        sport = packet[TCP].sport
        dport = packet[TCP].dport
        flags = packet[TCP].flags
        print(f'{src}:{sport} -> {dst}:{dport}  flags={flags}')

# Capture 10 packets on interface eth0
# sniff(iface='eth0', prn=packet_callback, count=10)
'''
print('Packet sniffer concept code (requires scapy + root):')
print(SNIFFER_CONCEPT)
print('Why root? The OS does not let unprivileged programs see other processes traffic.')
print('This is an important OS security boundary.')


### Log Analysis: Parse Apache Access Log


In [ ]:
import re
from collections import Counter

# Sample Apache Combined Log Format entries
sample_log = """192.168.1.10 - - [01/Jun/2024:10:01:01 +0000] "GET /index.html HTTP/1.1" 200 1234
10.0.0.5 - - [01/Jun/2024:10:01:05 +0000] "GET /admin HTTP/1.1" 404 512
10.0.0.5 - - [01/Jun/2024:10:01:06 +0000] "GET /admin/ HTTP/1.1" 404 512
10.0.0.5 - - [01/Jun/2024:10:01:07 +0000] "POST /login HTTP/1.1" 401 256
192.168.1.10 - - [01/Jun/2024:10:01:08 +0000] "GET /about HTTP/1.1" 200 4096
172.16.0.99 - - [01/Jun/2024:10:01:09 +0000] "GET /secret HTTP/1.1" 404 512
10.0.0.5 - - [01/Jun/2024:10:01:10 +0000] "GET /wp-admin HTTP/1.1" 404 512
10.0.0.5 - - [01/Jun/2024:10:01:11 +0000] "GET /.env HTTP/1.1" 404 512
"""

# Regex to parse Apache log line
LOG_RE = re.compile(
    r'(?P<ip>\d+\.\d+\.\d+\.\d+).*?"(?P<method>\w+) (?P<path>\S+).*?" (?P<status>\d{3})'
)

def analyze_log(log_text: str):
    entries   = []
    for line in log_text.strip().splitlines():
        m = LOG_RE.search(line)
        if m:
            entries.append(m.groupdict())

    # Count 404s by IP
    not_found = [e['ip'] for e in entries if e['status'] == '404']
    counter   = Counter(not_found)

    print('404 errors by IP (potential scanner):')
    for ip, cnt in counter.most_common():
        print(f'  {ip:15s}: {cnt} requests')

    # Suspicious paths
    suspicious = ['.env', 'wp-admin', 'admin', '.git', 'phpMyAdmin']
    print('\nSuspicious path requests:')
    for e in entries:
        if any(s in e['path'] for s in suspicious):
            print(f'  {e["ip"]:15s} -> {e["path"]} [{e["status"]}]')

analyze_log(sample_log)


### Student Challenge: Network Scanner

Write a function that pings each host in 192.168.1.0/24 to find
which ones are alive. This is a basic 'host discovery' sweep.

**Important:** Only run this on networks you own or have permission to scan.


In [ ]:
# ============ STUDENT EXERCISE ============
# TODO: Implement a subnet scanner using ping
#
# Use the ping_host() function defined earlier.
# Use threading to scan multiple hosts concurrently.
#
# def scan_subnet(base_ip: str, start: int = 1, end: int = 254) -> list:
#     """Ping sweep a /24 subnet. Returns list of alive IPs."""
#     pass
#
# Example call (DO NOT run on real networks without permission):
# alive_hosts = scan_subnet('192.168.1', 1, 10)
# print('Alive hosts:', alive_hosts)
#
# Expected: returns list of IPs that respond to ping
# ==========================================

import threading

def scan_subnet(base_ip: str, start: int = 1, end: int = 10) -> list:
    """Ping sweep a subnet range. Returns list of alive IPs."""
    # TODO: implement this
    # Hint: loop from start to end, build IP like f'{base_ip}.{i}'
    #       use threading for concurrent pings
    pass

# Safe test: ping loopback and localhost only
print('Testing ping on localhost:')
print(f'  127.0.0.1 alive: {ping_host("127.0.0.1")}')
print('\nTo scan a subnet (with permission):')
print('  alive = scan_subnet("192.168.1", 1, 10)')
